In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from scipy.signal import correlate2d
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import pandas as pd
from pathlib import Path

tf.keras.utils.set_random_seed(49)
# 1
matrix=np.random.randint(-5,6,(3,3)).astype(np.float32)
kernel=np.array([[1,0,-1],[1,0,-1],[1,0,-1]],dtype=np.float32)
padded=np.pad(matrix,1)
conv=correlate2d(padded,kernel,mode='valid')
relu=np.maximum(conv,0)
sigmoid=1/(1+np.exp(-conv))
windows=np.lib.stride_tricks.sliding_window_view(relu,(2,2))
print('Matrix:',matrix,'Padding:',padded,'Convolution:',conv,'ReLU:',relu,'Sigmoid:',sigmoid,sep='\n')
print('Max pooling:',windows.max(axis=(-2,-1)),'Min pooling:',windows.min(axis=(-2,-1)),'Global average:',relu.mean(),sep='\n')
x=tf.constant(relu[None,:,:,None])
flat=layers.Flatten()(x)
dense=layers.Dense(4)(flat)
bn=layers.BatchNormalization()
# Use a batch of distinct matrices for training-mode normalization.
batch=tf.concat([dense,dense+1,dense-1],axis=0)
normalized=bn(batch,training=True)
dropped=layers.Dropout(.3)(normalized,training=True)
print('Flatten:',flat.numpy(),'Dense:',dense.numpy(),'Batch normalization:',normalized.numpy(),'Dropout:',dropped.numpy(),sep='\n')
def early_stop():
    return keras.callbacks.EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)
# Demonstrate early stopping on a small illustrative regression task.
demo=keras.Sequential([layers.Input((9,)),layers.Dense(4,activation='relu'),layers.Dense(1)])
demo.compile(optimizer='adam',loss='mse')
demo_x=np.random.normal(size=(40,9)).astype('float32')
demo_y=demo_x.mean(axis=1)
demo_history=demo.fit(demo_x,demo_y,validation_split=.25,epochs=30,callbacks=[early_stop()],verbose=0)
print('Demo epochs:',len(demo_history.history['loss']))

In [ ]:
# 2
# Set this path to an extracted Tiny ImageNet dataset if already available.
root=Path('tiny-imagenet-200')
if not (root/'train').exists():
    archive=keras.utils.get_file('tiny-imagenet-200.zip',origin='https://cs231n.stanford.edu/tiny-imagenet-200.zip',extract=True)
    candidates=list(Path(archive).parent.rglob('tiny-imagenet-200'))
    root=next((p for p in candidates if (p/'train').is_dir()),root)
classes=sorted(p.name for p in (root/'train').iterdir() if p.is_dir() and len(list((p/'images').glob('*.JPEG')))>=200)[:3]
assert len(classes)==3
rng=np.random.default_rng(49)
X,y=[],[]
for label,cls in enumerate(classes):
    paths=sorted((root/'train'/cls/'images').glob('*.JPEG'))
    for path in rng.choice(paths,200,replace=False):
        X.append(keras.utils.img_to_array(keras.utils.load_img(path,target_size=(64,64))))
        y.append(label)
X=np.array(X,dtype=np.float32); y=np.array(y)
X_train,X_temp,y_train,y_temp=train_test_split(X,y,test_size=.30,stratify=y,random_state=49)
X_val,X_test,y_val,y_test=train_test_split(X_temp,y_temp,test_size=.50,stratify=y_temp,random_state=49)
print(classes, X_train.shape,X_val.shape,X_test.shape)
custom=keras.Sequential([
    layers.Input((64,64,3)),layers.Rescaling(1./255),
    layers.Conv2D(32,3,padding='same'),layers.BatchNormalization(),layers.ReLU(),layers.MaxPooling2D(),
    layers.Conv2D(64,3,padding='same'),layers.BatchNormalization(),layers.ReLU(),layers.MaxPooling2D(),
    layers.Conv2D(128,3,padding='same',activation='relu'),layers.GlobalAveragePooling2D(),
    layers.Dense(64,activation='relu'),layers.Dropout(.4),layers.Dense(3,activation='softmax')])
def train(model):
    model.compile(optimizer=keras.optimizers.Adam(1e-3),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    return model.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=30,batch_size=32,callbacks=[early_stop()],verbose=1)
history=train(custom)
plt.plot(history.history['loss'],label='Training loss')
plt.plot(history.history['val_loss'],label='Validation loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.show()
print('Compare loss curves: rising validation loss with falling training loss suggests overfitting; persistently high losses suggest underfitting.')

In [ ]:
# 3
from tensorflow.keras.applications import VGG16, MobileNet, ResNet50
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_pre
from tensorflow.keras.applications.mobilenet import preprocess_input as mobile_pre
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_pre

def transfer_model(builder,preprocess):
    base=builder(weights='imagenet',include_top=False,input_shape=(96,96,3))
    base.trainable=False
    inputs=keras.Input((64,64,3))
    x=layers.Resizing(96,96)(inputs)
    x=layers.Lambda(preprocess)(x)
    x=base(x,training=False)
    x=layers.GlobalAveragePooling2D()(x)
    x=layers.Dropout(.3)(x)
    outputs=layers.Dense(3,activation='softmax')(x)
    return keras.Model(inputs,outputs)

def evaluate(name,model):
    pred=np.argmax(model.predict(X_test,verbose=0),axis=1)
    precision,recall,f1,_=precision_recall_fscore_support(y_test,pred,average='macro',zero_division=0)
    return {'Model':name,'Accuracy':accuracy_score(y_test,pred),'Precision':precision,'Recall':recall,'F1':f1}
results=[evaluate('Custom CNN',custom)]
for name,builder,pre in [('VGG16',VGG16,vgg_pre),('MobileNet',MobileNet,mobile_pre),('ResNet50',ResNet50,resnet_pre)]:
    model=transfer_model(builder,pre)
    train(model)
    results.append(evaluate(name,model))
comparison=pd.DataFrame(results).set_index('Model')
print(comparison.round(4))
comparison.plot(kind='bar',ylim=(0,1),figsize=(10,5)); plt.ylabel('Score'); plt.xticks(rotation=0); plt.tight_layout(); plt.show()